# Notebook 12 – Model Selection

## Introduction to Model Selection
Model selection is the process of comparing different machine learning algorithms to determine which one provides the best performance for a specific dataset and business goal.

### Evaluation Criteria:
1. **Performance:** Accuracy, F1, RMSE, etc.
2. **Generalization:** How well it performs on unseen data (measured via Cross Validation).
3. **Training Time:** Time taken to fit the model.
4. **Prediction Time (Latency):** Time taken to produce a prediction for a single sample.
5. **Complexity:** Number of parameters, depth of trees.
6. **Interpretability:** How easy it is to explain *why* a prediction was made (e.g., Decision Tree is high, Neural Net is low).
7. **Resource Requirements:** Memory and CPU/GPU usage.

In [1]:
import numpy as np
import pandas as pd
import time
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score

# --- CLASSIFICATION COMPARISON ---
print("--- Classification Model Selection ---")
X_c, y_c = make_classification(n_samples=1000, n_features=20, random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

clf_models = {
    "Logistic Regression": LogisticRegression(),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "SVM": SVC(),
    "Gradient Boosting": GradientBoostingClassifier()
}

clf_results = []
for name, model in clf_models.items():
    start_train = time.time()
    model.fit(X_train_c, y_train_c)
    train_time = time.time() - start_train
    
    start_pred = time.time()
    y_pred = model.predict(X_test_c)
    pred_time = time.time() - start_pred
    
    clf_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_c, y_pred),
        "Train Time": train_time,
        "Pred Time": pred_time
    })

df_clf = pd.DataFrame(clf_results).sort_values(by="Accuracy", ascending=False)
print(df_clf)

--- Classification Model Selection ---
                 Model  Accuracy  Train Time  Pred Time
5    Gradient Boosting     0.910    0.766796   0.001058
3        Random Forest     0.885    0.539444   0.039834
0  Logistic Regression     0.855    0.061181   0.001846
2        Decision Tree     0.850    0.023117   0.000669
4                  SVM     0.845    0.070069   0.012553
1                  KNN     0.810    0.006490   2.082874


In [2]:
# --- REGRESSION COMPARISON ---
print("\n--- Regression Model Selection ---")
X_r, y_r = make_regression(n_samples=1000, n_features=20, noise=0.1, random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_r, y_r, test_size=0.2, random_state=42)

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

reg_models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor()
}

reg_results = []
for name, model in reg_models.items():
    start_train = time.time()
    model.fit(X_train_r, y_train_r)
    train_time = time.time() - start_train
    
    start_pred = time.time()
    y_pred = model.predict(X_test_r)
    pred_time = time.time() - start_pred
    
    reg_results.append({
        "Model": name,
        "R2 Score": r2_score(y_test_r, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test_r, y_pred)),
        "Train Time": train_time,
        "Pred Time": pred_time
    })

df_reg = pd.DataFrame(reg_results).sort_values(by="R2 Score", ascending=False)
print(df_reg)


--- Regression Model Selection ---
               Model  R2 Score        RMSE  Train Time  Pred Time
0  Linear Regression  1.000000    0.106901    0.039052   0.000690
1              Ridge  0.999998    0.275819    0.011559   0.000632
2              Lasso  0.999723    3.273315    0.003147   0.000412
5  Gradient Boosting  0.920650   55.421597    0.632065   0.000791
4      Random Forest  0.820174   83.431553    1.152818   0.025359
3      Decision Tree  0.471047  143.091113    0.018852   0.000483


## Final Comparison and Conclusion

| Model | Performance | Interpretability | Complexity | Resource Need |
| :--- | :--- | :--- | :--- | :--- |
| **Linear/Log Reg** | Low/Medium | Very High | Low | Very Low |
| **Decision Tree** | Medium | High | Low | Low |
| **Random Forest** | High | Low/Medium | Medium | Medium |
| **SVM** | High | Low | Medium | High |
| **Gradient Boosting** | Very High | Low | High | High |

### How to choose the right model?
1. **Start Simple:** Always start with a baseline (e.g., Logistic/Linear Regression).
2. **Check Data Size:** For very large datasets, prefer LightGBM or Random Forest over SVM.
3. **Business Need:** If you need to explain the decision to a stakeholder $\to$ Decision Tree or Logistic Regression. If you only care about the absolute highest accuracy $\to$ Gradient Boosting / XGBoost.
4. **Latency:** If predictions must be made in milliseconds $\to$ Linear models or small Trees.